# Analisis estadistico y predictivo REM20

Fase 4: descomposicion temporal, modelo predictivo de indice ocupacional y clustering de establecimientos.

In [1]:
import os
import pathlib
import warnings
import urllib.parse

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import sqlalchemy
from dotenv import load_dotenv
import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from statsmodels.tsa.seasonal import STL

warnings.filterwarnings("ignore")

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except Exception:
    sns.set_theme(style="whitegrid")

def find_root(start: pathlib.Path) -> pathlib.Path:
    for candidate in [start, *start.parents]:
        if (candidate / ".env").exists():
            return candidate
    raise FileNotFoundError("No se encontro .env al subir desde el notebook")

ROOT = find_root(pathlib.Path.cwd().resolve())
GRAF = ROOT / "data" / "processed" / "graficos"
MODELOS = ROOT / "data" / "processed" / "modelos"
GRAF.mkdir(parents=True, exist_ok=True)
MODELOS.mkdir(parents=True, exist_ok=True)

load_dotenv(ROOT / ".env")
db_user = urllib.parse.quote_plus(os.getenv("DB_USER"))
db_password = urllib.parse.quote_plus(os.getenv("DB_PASSWORD"))
db_url = (
    f"postgresql+psycopg2://{db_user}:{db_password}@"
    f"{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)
engine = sqlalchemy.create_engine(db_url, pool_pre_ping=True)

FUENTE = "Fuente: MINSAL REM20 2014-2026"

def guardar_grafico(fig, nombre: str) -> None:
    fig.text(0.99, 0.01, FUENTE, ha="right", va="bottom", fontsize=8, style="italic", color="gray")
    fig.tight_layout()
    fig.savefig(GRAF / f"{nombre}.png", dpi=120, bbox_inches="tight")

df = pd.read_sql("SELECT * FROM rem20.indicadores", engine)

numeric_cols = [
    "periodo", "mes", "codigo_establecimiento", "cod_area_funcional", "tipo_pertenencia", "cod_sss",
    "dias_camas_ocupadas", "dias_camas_disponibles", "dias_estada", "numero_egresos",
    "egresos_fallecidos", "traslados", "indice_ocupacional", "promedio_camas_disponible",
    "promedio_dias_estada", "letalidad", "indice_rotacion",
]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print(f"Filas cargadas desde rem20.indicadores: {len(df):,}")

Filas cargadas desde rem20.indicadores: 165,232


## 1. Descomposicion STL

Se construye una serie mensual nacional usando el promedio del indice ocupacional y se separa en tendencia, componente estacional y residuo.

In [2]:
df_stl = df.copy()
df_stl["fecha"] = pd.to_datetime(dict(year=df_stl["periodo"], month=df_stl["mes"], day=1))
serie = df_stl.groupby("fecha")["indice_ocupacional"].mean().sort_index()
serie = serie.asfreq("MS")
if serie.isna().any():
    serie = serie.interpolate(method="linear")

stl = STL(serie, period=12, robust=True).fit()

fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
axes[0].plot(stl.observed.index, stl.observed.values, color="#1f77b4")
axes[0].set_title("Serie observada")
axes[0].set_ylabel("Indice")
axes[1].plot(stl.trend.index, stl.trend.values, color="#2ca02c")
axes[1].set_title("Tendencia")
axes[1].set_ylabel("Indice")
axes[2].plot(stl.seasonal.index, stl.seasonal.values, color="#ff7f0e")
axes[2].set_title("Componente estacional")
axes[2].set_ylabel("Indice")
axes[3].plot(stl.resid.index, stl.resid.values, color="#d62728")
axes[3].set_title("Residuo")
axes[3].set_ylabel("Indice")
axes[3].set_xlabel("Fecha")
fig.suptitle("Descomposicion STL del indice ocupacional nacional", fontsize=14)
guardar_grafico(fig, "05_stl_descomposicion")
plt.close(fig)

# Comparacion de variabilidad de las componentes.
# Nota: la componente estacional de STL oscila alrededor de 0 (media ~= 0), por lo que
# su CV clasico (std/media propia) es degenerado. Para que ambas componentes sean
# comparables se usa un denominador COMUN: la media de la serie observada (nivel del indice).
media_serie = serie.mean()
cv_trend = stl.trend.std() / media_serie
cv_seasonal = stl.seasonal.std() / media_serie
dominante = "tendencia" if cv_trend > cv_seasonal else "estacionalidad"

seasonal_by_month = stl.seasonal.groupby(stl.seasonal.index.month).mean()
mes_peak = int(seasonal_by_month.idxmax())

print(f"Media de la serie observada: {media_serie:.2f}")
print(f"Variabilidad relativa de la tendencia (std/media): {cv_trend:.4f}")
print(f"Variabilidad relativa de la estacionalidad (std/media): {cv_seasonal:.4f}")
print(f"Componente con mayor variabilidad relativa: {dominante}")
print(f"Mes con mayor componente estacional promedio: {mes_peak}")

Media de la serie observada: 64.15
Variabilidad relativa de la tendencia (std/media): 0.0357
Variabilidad relativa de la estacionalidad (std/media): 0.0328
Componente con mayor variabilidad relativa: tendencia
Mes con mayor componente estacional promedio: 8


## 2. Modelo predictivo

El modelo usa lags calculados por cada serie establecimiento-area para evitar fuga de datos entre series. La particion es temporal: entrenamiento hasta 2023 y prueba en 2024-2025.

In [3]:
dfm = df.copy()
dfm = dfm.sort_values(["codigo_establecimiento", "cod_area_funcional", "periodo", "mes"])
dfm["fecha"] = pd.to_datetime(dict(year=dfm["periodo"], month=dfm["mes"], day=1))

grupo_cols = ["codigo_establecimiento", "cod_area_funcional"]
dfm = dfm.groupby(grupo_cols, group_keys=False).filter(lambda g: len(g) >= 24).copy()
dfm = dfm.sort_values(grupo_cols + ["fecha"])

dfm["mes_sin"] = np.sin(2 * np.pi * dfm["mes"] / 12)
dfm["mes_cos"] = np.cos(2 * np.pi * dfm["mes"] / 12)

g = dfm.groupby(grupo_cols, group_keys=False)["indice_ocupacional"]
dfm["lag_1"] = g.transform(lambda s: s.shift(1))
dfm["lag_12"] = g.transform(lambda s: s.shift(12))
dfm["rolling_3"] = g.transform(lambda s: s.shift(1).rolling(window=3).mean())

label_encoder = LabelEncoder()
dfm["area_funcional_cod"] = label_encoder.fit_transform(dfm["area_funcional"].astype(str))
dfm["cod_sss"] = pd.to_numeric(dfm["cod_sss"], errors="coerce")

features = ["mes_sin", "mes_cos", "lag_1", "lag_12", "rolling_3", "area_funcional_cod", "cod_sss"]
target = "indice_ocupacional"
df_model = dfm.dropna(subset=features + [target]).copy()

train = df_model[df_model["periodo"] <= 2023]
test = df_model[df_model["periodo"].isin([2024, 2025])]

X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]

rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Filas entrenamiento: {len(train):,}")
print(f"Filas prueba 2024-2025: {len(test):,}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"R2: {r2:.4f}")

importancias = pd.Series(rf.feature_importances_, index=features).sort_values()
fig, ax = plt.subplots(figsize=(12, 6))
importancias.plot(kind="barh", ax=ax, color="#2a6f97")
ax.set_title("Importancia de variables - Random Forest")
ax.set_xlabel("Importancia")
ax.set_ylabel("Variable")
guardar_grafico(fig, "06_feature_importance")
plt.close(fig)

# compress=3: el RandomForest sin podar es muy grande (~1 GB sin comprimir);
# la compresion reduce el archivo en disco sin alterar el modelo ni las metricas.
joblib.dump(
    {"model": rf, "features": features, "label_encoder_area_funcional": label_encoder},
    MODELOS / "rf_indice_ocupacional.pkl",
    compress=3,
)

Filas entrenamiento: 116,443
Filas prueba 2024-2025: 26,185
RMSE: 19.9484
MAE: 8.1326
R2: 0.6364


['C:\\Users\\matia\\Memoria\\proyectos\\Analisis_Hospitalario\\data\\processed\\modelos\\rf_indice_ocupacional.pkl']

In [4]:
top5_est = (
    df.groupby("codigo_establecimiento")["numero_egresos"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
    .index
)

pred_2026 = df_model[(df_model["periodo"] == 2026) & (df_model["codigo_establecimiento"].isin(top5_est))].copy()
pred_2026["predicho"] = rf.predict(pred_2026[features])

fig, ax = plt.subplots(figsize=(12, 6))
if pred_2026.empty:
    ax.text(0.5, 0.5, "Sin datos 2026 disponibles para los establecimientos seleccionados", ha="center", va="center")
    ax.set_axis_off()
else:
    resumen_2026 = (
        pred_2026.groupby("fecha")[["indice_ocupacional", "predicho"]]
        .mean()
        .reset_index()
    )
    ax.plot(resumen_2026["fecha"], resumen_2026["indice_ocupacional"], marker="o", label="Real 2026")
    ax.plot(resumen_2026["fecha"], resumen_2026["predicho"], marker="o", label="Predicho 2026")
    ax.set_title("Indice ocupacional real vs predicho en 2026 - top 5 establecimientos por egresos")
    ax.set_xlabel("Fecha")
    ax.set_ylabel("Indice ocupacional promedio")
    ax.legend()
guardar_grafico(fig, "06_pred_2026")
plt.close(fig)

pred_2026[["fecha", "codigo_establecimiento", "establecimiento", "cod_area_funcional", "indice_ocupacional", "predicho"]].head(20)

,fecha,codigo_establecimiento,establecimiento,cod_area_funcional,indice_ocupacional,predicho
161919,2026-01-01,113100,"Hospital Barros Luco Trudeau (Santiago, San Mi...",330,41.47,40.0014
161920,2026-02-01,113100,"Hospital Barros Luco Trudeau (Santiago, San Mi...",330,34.95,45.2388
161921,2026-03-01,113100,"Hospital Barros Luco Trudeau (Santiago, San Mi...",330,40.09,37.0832
161922,2026-04-01,113100,"Hospital Barros Luco Trudeau (Santiago, San Mi...",330,43.10,38.9513
161923,2026-05-01,113100,"Hospital Barros Luco Trudeau (Santiago, San Mi...",330,39.17,37.4871
161878,2026-01-01,113100,"Hospital Barros Luco Trudeau (Santiago, San Mi...",401,89.62,82.7401
161879,2026-02-01,113100,"Hospital Barros Luco Trudeau (Santiago, San Mi...",401,85.29,84.1279
161880,2026-03-01,113100,"Hospital Barros Luco Trudeau (Santiago, San Mi...",401,87.81,85.9269
161881,2026-04-01,113100,"Hospital Barros Luco Trudeau (Santiago, San Mi...",401,88.99,88.4430
161882,2026-05-01,113100,"Hospital Barros Luco Trudeau (Santiago, San Mi...",401,91.59,87.9974


## 3. Clustering de establecimientos

Se usa `k=4` como decision inicial documentada para segmentar perfiles hospitalarios. El grafico del codo queda disponible para que el orquestador ajuste el numero de clusters si observa una alternativa mas clara.

In [5]:
cluster_features = ["indice_ocupacional", "promedio_dias_estada", "letalidad", "indice_rotacion"]
df_cluster = (
    df.groupby("codigo_establecimiento")
    .agg(
        establecimiento=("establecimiento", "first"),
        indice_ocupacional=("indice_ocupacional", "mean"),
        promedio_dias_estada=("promedio_dias_estada", "mean"),
        letalidad=("letalidad", "mean"),
        indice_rotacion=("indice_rotacion", "mean"),
    )
    .dropna(subset=cluster_features)
    .reset_index()
)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_cluster[cluster_features])

ks = list(range(2, 11))
inercias = []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(X_scaled)
    inercias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(ks, inercias, marker="o", color="#6a4c93")
ax.set_title("Metodo del codo para KMeans")
ax.set_xlabel("Numero de clusters (k)")
ax.set_ylabel("Inercia")
ax.set_xticks(ks)
guardar_grafico(fig, "07_codo_kmeans")
plt.close(fig)

k_elegido = 4
kmeans = KMeans(n_clusters=k_elegido, n_init=10, random_state=42)
df_cluster["cluster"] = kmeans.fit_predict(X_scaled)

pca = PCA(n_components=2, random_state=42)
componentes = pca.fit_transform(X_scaled)
df_cluster["pc1"] = componentes[:, 0]
df_cluster["pc2"] = componentes[:, 1]

fig, ax = plt.subplots(figsize=(12, 6))
sns.scatterplot(data=df_cluster, x="pc1", y="pc2", hue="cluster", palette="tab10", s=70, ax=ax)
ax.set_title("Clusters de establecimientos proyectados con PCA")
ax.set_xlabel("Componente principal 1")
ax.set_ylabel("Componente principal 2")

for cluster_id in sorted(df_cluster["cluster"].unique()):
    idx_cluster = np.where(df_cluster["cluster"].to_numpy() == cluster_id)[0]
    distancias = np.linalg.norm(X_scaled[idx_cluster] - kmeans.cluster_centers_[cluster_id], axis=1)
    cercanos = idx_cluster[np.argsort(distancias)[:3]]
    for idx in cercanos:
        row = df_cluster.iloc[idx]
        etiqueta = str(row["establecimiento"])[:32]
        ax.annotate(etiqueta, (row["pc1"], row["pc2"]), fontsize=8, xytext=(4, 4), textcoords="offset points")

guardar_grafico(fig, "07_clusters_pca")
plt.close(fig)

caracterizacion = (
    df_cluster.groupby("cluster")
    .agg(
        establecimientos=("codigo_establecimiento", "count"),
        indice_ocupacional=("indice_ocupacional", "mean"),
        promedio_dias_estada=("promedio_dias_estada", "mean"),
        letalidad=("letalidad", "mean"),
        indice_rotacion=("indice_rotacion", "mean"),
    )
    .round(3)
)
print(caracterizacion)

joblib.dump(
    {"model": kmeans, "scaler": scaler, "features": cluster_features, "k": k_elegido},
    MODELOS / "kmeans_establecimientos.pkl",
    compress=3,
)

caracterizacion

         establecimientos  indice_ocupacional  promedio_dias_estada  \
cluster                                                               
0                     120              43.664                 9.797   
1                      82              73.389                17.814   
2                       4              81.755               343.189   
3                       2             130.150                 4.493   

         letalidad  indice_rotacion  
cluster                              
0            2.657            2.199  
1           10.763            2.723  
2            2.161            1.714  
3            1.606           30.351  


,establecimientos,indice_ocupacional,promedio_dias_estada,letalidad,indice_rotacion
cluster,,,,,
0,120,43.664,9.797,2.657,2.199
1,82,73.389,17.814,10.763,2.723
2,4,81.755,343.189,2.161,1.714
3,2,130.150,4.493,1.606,30.351


### Interpretacion de clusters (k=4)

Caracterizacion sobre el promedio historico por establecimiento de los 4 indicadores:

- **Cluster 0 — Baja complejidad / baja demanda (120 establecimientos, ~57%):** ocupacion baja
  (~43,7%), estada corta (~9,8 dias), letalidad baja (~2,7%) y rotacion baja (~2,2). Es el grupo
  mayoritario; concentra hospitales pequenos y comunitarios de menor presion asistencial.
- **Cluster 1 — Alta complejidad / agudos (82 establecimientos, ~39%):** ocupacion alta (~73,4%),
  estada media (~17,8 dias) y **letalidad alta (~10,8%)**. Perfil de hospitales de mayor
  complejidad y pacientes mas graves, con la mayor presion asistencial del sistema.
- **Cluster 2 — Larga estadia / psiquiatricos (4 establecimientos, ~2%):** estada extraordinaria
  (~343 dias), ocupacion alta (~81,8%) y rotacion muy baja (~1,7). Lo integran hospitales
  psiquiatricos (El Peral, Jose Horwitz, Philippe Pinel). La letalidad baja (~2,2%) confirma que
  la larga estadia responde al modelo de atencion, no a gravedad clinica.
- **Cluster 3 — Altisima rotacion / corta estadia (2 establecimientos, ~1%):** rotacion atipica
  (~30,4), estada muy corta (~4,5 dias) y ocupacion >100% por camas prestadas entre servicios.
  Perfil marginal y atipico (Hospital de Llanquihue, CESFAM Rio Negro).
